# Module 04: Load Balancing Algorithms Health Probes — Interactive Laboratory

Every cell below runs the module's **real** implementation from
`project_solution/load_balancer.py`. Nothing here prints a claim it has not verified.

What you will do:

1. Load the engine and inspect what it actually exports.
2. Run its primary workflow and check the assertions that define correctness.
3. **Commit to a prediction**, then run the cell that tests it.
4. Measure a property rather than asserting one.
5. Fix a deliberately broken cell in place.

> The code in cells 4, 6 and 8 is lifted from this module's own test suite, so it
> cannot drift from the implementation. If the API changes, those tests fail
> first and this notebook is regenerated from them.


## 1. Load the engine and introspect it

Rather than trusting a hardcoded list of class names, ask the module what it
actually contains.


In [ ]:
import inspect
import sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve() / 'project_solution'))
import load_balancer

classes = [n for n, o in inspect.getmembers(load_balancer, inspect.isclass)
           if o.__module__ == 'load_balancer']
functions = [n for n, o in inspect.getmembers(load_balancer, inspect.isfunction)
             if o.__module__ == 'load_balancer']

print('module   : load_balancer')
print(f'classes  : {classes}')
print(f'functions: {functions}')
print()
for name in classes:
    obj = getattr(load_balancer, name)
    try:
        sig = inspect.signature(obj.__init__)
        params = [p for p in sig.parameters if p != 'self']
    except (TypeError, ValueError):
        params = ['<builtin>']
    print(f'  {name}({", ".join(params)})')

## 2. Baseline: Round robin distribution

This is the module's own `test_round_robin_distribution` — real instantiation, real calls, real
assertions. If it runs clean, the property it encodes holds.


In [ ]:
from load_balancer import (
    BackendServer,
    LeastConnectionsStrategy,
    LoadBalancer,
    RoundRobinStrategy,
    WeightedRoundRobinStrategy,
)

lb = LoadBalancer(strategy=RoundRobinStrategy())
s1 = BackendServer(server_id="s1", host="10.0.0.1", port=8080)
s2 = BackendServer(server_id="s2", host="10.0.0.2", port=8080)
s3 = BackendServer(server_id="s3", host="10.0.0.3", port=8080)

for s in [s1, s2, s3]:
    lb.register_server(s)

routed = [lb.route_request().server_id for _ in range(6)]
assert routed == ["s1", "s2", "s3", "s1", "s2", "s3"]

print('PASSED: test_round_robin_distribution')

## 3. 🔮 Prediction — commit before you run

Round-robin sends equal request *counts* to every backend. Predict what happens to p99 latency when one backend is 5x slower but still healthy.

Write your answer down. An uncommitted guess teaches nothing, because you will
retro-fit it to whatever the next cell prints.

The next cell runs `test_weighted_round_robin_distribution`, which tests exactly this property.


In [ ]:
lb = LoadBalancer(strategy=WeightedRoundRobinStrategy())
s1 = BackendServer(server_id="s1", host="10.0.0.1", port=8080, weight=3)
s2 = BackendServer(server_id="s2", host="10.0.0.2", port=8080, weight=1)

lb.register_server(s1)
lb.register_server(s2)

routed = [lb.route_request().server_id for _ in range(8)]
# Over 8 requests with 3:1 weights, s1 should receive 6 and s2 should receive 2
assert routed.count("s1") == 6
assert routed.count("s2") == 2

print('PASSED: test_weighted_round_robin_distribution')

## 4. Measure it: Least connections strategy

An assertion tells you a property holds. A measurement tells you *how much*.
This cell runs `test_least_connections_strategy` and times it.


In [ ]:
import time

_t0 = time.perf_counter()

lb = LoadBalancer(strategy=LeastConnectionsStrategy())
s1 = BackendServer(server_id="s1", host="10.0.0.1", port=8080, active_connections=5)
s2 = BackendServer(server_id="s2", host="10.0.0.2", port=8080, active_connections=1)
s3 = BackendServer(server_id="s3", host="10.0.0.3", port=8080, active_connections=3)

for s in [s1, s2, s3]:
    lb.register_server(s)

# First route goes to s2 (1 connection)
routed1 = lb.route_request()
assert routed1.server_id == "s2"
assert s2.active_connections == 2

# Second route also goes to s2 (now tied at 2, but s3 has 3)
routed2 = lb.route_request()
assert routed2.server_id == "s2"
assert s2.active_connections == 3

_elapsed = (time.perf_counter() - _t0) * 1000
print('PASSED: test_least_connections_strategy')
print(f'wall clock: {_elapsed:.2f} ms')

## 5. 🛠️ Fix this cell — it is deliberately broken

The cell below asserts something **false** about the real object. Read the
failure, work out the true value from the module's actual behaviour, and correct
the expected number.

Do not delete the assertion. The point is to make it pass by knowing the answer.


In [ ]:
# DELIBERATELY BROKEN - fix the expected value below.
# Hint: print the real value first, then decide what the assertion should say.

exports = [n for n in dir(load_balancer) if not n.startswith('_')]
print(f'actual export count: {len(exports)}')
print(f'actual exports     : {exports}')

EXPECTED_EXPORT_COUNT = 999      # <-- wrong on purpose. Replace it.

assert len(exports) == EXPECTED_EXPORT_COUNT, (
    f'expected {EXPECTED_EXPORT_COUNT} exports, found {len(exports)}. '
    'Read the printed value above and correct the constant.'
)
print('Fixed - assertion now reflects reality.')

### 🎓 Key takeaways

1. Equal request counts do not mean equal load - least-connections beats round-robin under heterogeneous latency.
2. Passive health checks react faster than active ones but need traffic to work.
3. Sticky sessions trade balance for locality; know which you are buying.

---

**Continue with this module:**

- [README.md](README.md) — the mental model and failure modes
- [PROJECT_GUIDE.md](PROJECT_GUIDE.md) — build it yourself, in 3 tiers
- [starter/](starter/) — your stubs; run the tests from there to grade yourself
- [debug_lab/SYMPTOMS.md](debug_lab/SYMPTOMS.md) — diagnose planted bugs from the symptom
- [TROUBLESHOOTING_AND_EDGE_CASES.md](TROUBLESHOOTING_AND_EDGE_CASES.md) — real errors, real causes
- [SELF_ASSESSMENT_AND_CHALLENGES.md](SELF_ASSESSMENT_AND_CHALLENGES.md) — quiz and diagnostics
